# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete example for loading and exploring the FAIR<sup>2</sup> Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Authors: {dataset.metadata.author}")
print(f"Available record sets: {dataset.metadata.recordSet}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id` values.

Below we examine all record sets and enumerate their included fields, using each field's `@id`, `name`, and type.

In [ ]:
# List all record sets and their fields with @id references
all_record_sets = dataset.metadata.recordSet
print(f"Found {len(all_record_sets)} record sets.\n")

for rs in all_record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name','N/A')}")
    # Fields for this record set (if present)
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for f in fields:
            print(f"    - @id: {f['@id']}, name: {f.get('name','N/A')}, dataType: {f.get('dataType','N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset, we'll extract and examine the primary protein abundance table. _Replace the example `@id` below with the actual record set and field ids you want to explore._

In [ ]:
# Collect all record set @ids (for convenience)
record_set_ids = [rs['@id'] for rs in all_record_sets]

# Load all record sets' data into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
    print(f"Columns: {df.columns.tolist()}")

# For example, show head of the first record set
if len(record_set_ids) > 0:
    df = dataframes[record_set_ids[0]]
    print(f"\nPreview of first record set (@id={record_set_ids[0]}):")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filtering records, normalizing numeric fields, and optionally grouping the data. 

Below we select a numeric field (`MW`, i.e., molecular weight) and demonstrate filtering and normalization.

> **References to fields are always by their `@id` as defined in the Croissant schema.**

In [ ]:
# Choose the main protein table record set (replace with appropriate @id if your overview lists differently)
main_rs_id = record_set_ids[0]  # For this dataset, the first record set is typically the protein table

# Display available columns again
df = dataframes[main_rs_id]
print(f"Numeric columns in this record set:")
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(numeric_cols)

# Suppose the Croissant field for molecular weight has @id 'cr:MW'. (Update as appropriate from the overview cell above!)
numeric_field_id = None
for col in df.columns:
    lower = col.lower()
    if lower in ["mw", "molecularweight", "molecular_weight", "molecularweight(kda)"]:
        numeric_field_id = col
        break
if not numeric_field_id and numeric_cols:
    numeric_field_id = numeric_cols[0]  # fallback
    print(f"Using default numeric field: {numeric_field_id}")

# Filter and normalize values
threshold = df[numeric_field_id].mean() if numeric_field_id else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a common field (e.g. 'description' or 'sample')
group_field = None
for cand in ['description', 'group', 'sample', 'accession']:
    if cand in df.columns:
        group_field = cand
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    print(f"Grouped mean {numeric_field_id} values by {group_field}:")
    display(grouped_df.head())
else:
    print(f"No obvious group field found for this record set.")

## 5. Visualization
Visualize data distributions or relationships between two fields using the loaded DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the main numeric field
if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# Scatter plot vs another numeric field (e.g., peptide_count if it exists)
secondary_field = None
for cand in ['peptide_count', 'num_peptides', 'count']:
    if cand in df.columns:
        secondary_field = cand
        break
if numeric_field_id and secondary_field:
    plt.figure(figsize=(7,5))
    sns.scatterplot(data=df, x=numeric_field_id, y=secondary_field)
    plt.title(f"{numeric_field_id} vs {secondary_field}")
    plt.xlabel(numeric_field_id)
    plt.ylabel(secondary_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset via Croissant manifest using `mlcroissant`. We provided a data overview, extracted data into DataFrames, performed basic EDA, and visualized distributions. For detailed downstream analyses, refer to the dataset's documentation and Croissant metadata for full field definitions and processing suggestions.